# 03 可选 TARDIS 建模

这个 notebook 的定位刻意收窄。文献调研说明，稀疏光谱样本不适合过度建模；这里的 TARDIS 主要用于在分类、相位和速度检查之后，辅助谱线识别和定性比较谱形。

只有当目标有可用观测光谱，并且 `output/<target>/tardis/` 下已经有 TARDIS 输出时，才需要在 `tardis` 环境中运行这个 notebook。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TARGET = "SN2026jlm"
OUTPUT_DIR = PROJECT_ROOT / "output" / TARGET
SPECTRUM_DIR = OUTPUT_DIR / "spectrum"
TARDIS_DIR = OUTPUT_DIR / "tardis"

## 读取观测光谱和模拟光谱

In [ ]:
observed_files = sorted(SPECTRUM_DIR.glob("*.dat"))
simulated_files = sorted(TARDIS_DIR.glob("tardis_spectrum_*.dat"))

if not observed_files:
    raise FileNotFoundError(f"No observed *.dat spectra in {SPECTRUM_DIR}")
if not simulated_files:
    raise FileNotFoundError(f"No TARDIS spectra in {TARDIS_DIR}")

observed_path = observed_files[0]
simulated_path = simulated_files[0]
obs = np.loadtxt(observed_path)
sim = np.loadtxt(simulated_path)

wave_obs, flux_obs = obs[:, 0], obs[:, 1]
wave_sim, flux_sim = sim[:, 0], sim[:, 1]

print(f"观测光谱: {observed_path}")
print(f"模拟光谱: {simulated_path}")

## 归一化并比较谱形

In [ ]:
def normalize(flux):
    finite = np.isfinite(flux)
    scale = np.nanpercentile(np.abs(flux[finite]), 95) if finite.any() else 1.0
    return flux / scale if scale else flux

plt.figure(figsize=(10, 5))
plt.plot(wave_obs, normalize(flux_obs), lw=0.9, label="Observed WISeREP/BFOSC spectrum")
plt.plot(wave_sim, normalize(flux_sim), lw=0.9, label="TARDIS synthetic spectrum")
plt.xlim(3500, 9000)
plt.xlabel("Rest wavelength (Angstrom)")
plt.ylabel("Normalized flux / luminosity density")
plt.title(f"{TARGET}: qualitative TARDIS comparison")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 解释边界

这个比较只能用来讨论模型是否大体匹配谱线位置或连续谱形状。若没有进一步建模和不确定度分析，不应从一条稀疏光谱中给出强的抛射物质量、爆炸能量或丰度约束。